# AM Fundamentals

This notebook breaks the AM section out of both legacy notebooks. It keeps the core ideas together: envelope shape, sidebands, modulation index, and envelope detection.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


In [ ]:
VOICE_FILE = ROOT / "assets" / "local" / "my_voice.m4a"
WORK_FS = 96_000
PLAY_FS = 44_100

if VOICE_FILE.exists():
    raw_fs, raw_audio = load_audio(VOICE_FILE, normalize_audio=True)
    raw_audio = normalize(ensure_mono(raw_audio))
    raw_audio = raw_audio[: int(raw_fs * 5)]
    voice_work = normalize(resample_signal(raw_audio, raw_fs, WORK_FS))
    voice_play = normalize(resample_signal(raw_audio, raw_fs, PLAY_FS))
    print(f"Loaded {VOICE_FILE.name} at {raw_fs} Hz")
else:
    t_fallback = np.arange(0, 3.0, 1 / WORK_FS)
    voice_work = normalize(
        0.7 * np.sin(2 * np.pi * 220 * t_fallback)
        + 0.4 * np.sin(2 * np.pi * 440 * t_fallback)
        + 0.2 * np.sin(2 * np.pi * 880 * t_fallback)
    )
    voice_play = normalize(resample_signal(voice_work, WORK_FS, PLAY_FS))
    print(f"No local voice recording found at {VOICE_FILE}. Using a synthetic fallback.")

t_work = np.arange(len(voice_work)) / WORK_FS


## Envelope Modulation

AM multiplies a carrier by a shifted version of the message:

$$s_{AM}(t) = [1 + m x(t)] \cos(2\pi f_c t)$$

The envelope follows the message as long as the modulation index stays below 1.

In [ ]:
carrier_freq = 20_000
message = signal.sosfilt(signal.butter(5, [300, 3000], btype="band", fs=WORK_FS, output="sos"), voice_work)
message = normalize(message)
am_signal = am_modulate(message, carrier_freq=carrier_freq, fs=WORK_FS, mod_index=0.8)
am_demod = am_demodulate(am_signal, fs=WORK_FS)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
win = slice(0, 8000)
axes[0].plot(t_work[win] * 1000, message[win], color="tab:green")
axes[0].set_title("Message")
axes[0].set_xlabel("Time (ms)")
axes[1].plot(t_work[win] * 1000, am_signal[win], color="tab:blue", linewidth=0.6)
axes[1].set_title("AM Waveform")
axes[1].set_xlabel("Time (ms)")
plot_spectrum(am_signal, fs=WORK_FS, ax=axes[2], title="AM Spectrum")
axes[2].set_xlim(0, 30_000)
axes[2].set_ylim(-100, 5)
plt.tight_layout()

display(Markdown("**Demodulated AM audio**"))
display(audio_player(resample_signal(am_demod, WORK_FS, PLAY_FS), rate=PLAY_FS))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
audio_out = audio_output_widget()

def update_am(mod_index=0.8):
    am_signal = am_modulate(message, carrier_freq=carrier_freq, fs=WORK_FS, mod_index=mod_index)
    demod = am_demodulate(am_signal, fs=WORK_FS)
    axes[0].clear()
    axes[1].clear()
    win = slice(0, 6000)
    axes[0].plot(t_work[win] * 1000, am_signal[win], color="tab:blue", linewidth=0.6)
    axes[0].set_title(f"AM Waveform, m={mod_index:.2f}")
    axes[0].set_xlabel("Time (ms)")
    plot_spectrum(am_signal, fs=WORK_FS, ax=axes[1], title="AM Spectrum")
    axes[1].set_xlim(0, 30_000)
    axes[1].set_ylim(-100, 5)
    fig.canvas.draw_idle()
    refresh_audio_widget(audio_out, resample_signal(demod, WORK_FS, PLAY_FS), rate=PLAY_FS)

controls = widgets.interactive(
    update_am,
    mod_index=float_slider(min_value=0, max_value=1.5, step=0.05, value=0.8, description="m"),
)
display(controls, audio_out)


## What to Try

- Push `m` above 1 and listen for overmodulation distortion.
- Watch the sidebands stay symmetric around the carrier.
- Compare the message waveform to the AM envelope over a short zoomed-in segment.

## Key Takeaway

AM is visually intuitive because the message rides in the envelope, but that same amplitude dependence makes it more vulnerable to noise and impulsive interference.